# End-to-End v0.5.0 Feature Demo: HSC CMP vs CD14+ Monocyte

This notebook demonstrates all v0.5.0 features on real hematopoietic stem cell (HSC) data:

1. **Data loading** — Subset CMP and CD14+ Monocyte populations
2. **Archetype fitting** — Train K=4 models on each cell type
3. **Simplex regression** — Feature associations via Scheffé polynomials
4. **Pattern classification** — Categorize gene response patterns
5. **Archetype comparison** — MMD, feature similarity, Wald contrasts
6. **Flow matching** — CMP → Monocyte trajectory
7. **Visualization** — All new plot types

**Requirements:** PEACH v0.5.0, HSC.h5ad dataset

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["MKL_NUM_THREADS"] = "1"

import time
import warnings
import logging
import numpy as np
import scipy.sparse as sp
import pandas as pd
import anndata as ad

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=PendingDeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
logging.disable(logging.WARNING)

import torch
torch.set_num_threads(1)

import peach as pc

print(f"PEACH loaded. Torch threads: {torch.get_num_threads()}")

PEACH loaded. Torch threads: 1


## Configuration

In [2]:
HSC_PATH = "/Users/honkala/Desktop/cross_recons/data/HSC.h5ad"
K = 9
N_PCS = 13
MONO_SUBSAMPLE = 9000
TRAIN_EPOCHS = 150
FLOW_EPOCHS = 300
SEED = 42

## 1. Data Loading

Load the HSC dataset and extract CMP and CD14+ Monocyte subsets. We keep PCA coordinates and original indices for later sparse-X reload.

In [3]:
import gc

t0 = time.time()
adata_full = ad.read_h5ad(HSC_PATH)
print(f"Full dataset: {adata_full.shape[0]:,} cells")

rng = np.random.default_rng(SEED)
var_names = list(adata_full.var_names)

cell_data = {}
for name, ct, subsample in [
    ("CMP", "common myeloid progenitor", None),
    ("Mono", "CD14-positive monocyte", MONO_SUBSAMPLE),
]:
    mask = adata_full.obs["cell_type"] == ct
    idx = np.where(mask)[0]
    if subsample and len(idx) > subsample:
        idx = rng.choice(idx, size=subsample, replace=False)
        idx.sort()
    pca = adata_full.obsm["X_pca"][idx, :N_PCS].copy()
    obs = adata_full.obs.iloc[idx].copy().reset_index(drop=True)
    cell_data[name] = {"pca": pca, "obs": obs, "idx": idx}
    print(f"  {name}: {len(idx):,} cells, PCA: {pca.shape}")

del adata_full
gc.collect()
print(f"Data loading: {time.time()-t0:.1f}s")

Full dataset: 263,159 cells
  CMP: 9,366 cells, PCA: (9366, 13)
  Mono: 9,000 cells, PCA: (9000, 13)
Data loading: 32.2s


## 2. Archetype Fitting

Train K=4 archetypes on each cell type. We use a lightweight AnnData with a 1-column sparse placeholder for X during training (the model only uses `X_pca`), then rebuild with real sparse X afterward for regression.

In [ ]:
models = {}
for name in ["CMP", "Mono"]:
    t1 = time.time()
    
    # Lightweight AnnData — avoids OMP issues with large sparse X
    adata_train = ad.AnnData(
        X=sp.csr_matrix((len(cell_data[name]["pca"]), 1)),
        obs=cell_data[name]["obs"].copy(),
    )
    adata_train.obsm["X_pca"] = cell_data[name]["pca"]
    
    result = pc.tl.train_archetypal(
        adata_train, n_archetypes=K,
        n_epochs=TRAIN_EPOCHS, kld_weight=0.1, archetypal_weight=0.9,
        seed=SEED,
    )
    
    r2 = result.get("final_archetype_r2", None)
    r2_str = f"R2={r2:.3f}" if r2 else "R2=N/A"
    
    # Workflow 04 steps: weights, coordinates, assignment
    pc.tl.extract_archetype_weights(adata_train)
    pc.tl.archetypal_coordinates(adata_train, verbose=False)
    
    cell_data[name]["weights"] = adata_train.obsm["cell_archetype_weights"]
    cell_data[name]["uns"] = dict(adata_train.uns)
    cell_data[name]["arch_dist"] = adata_train.obsm["archetype_distances"]
    models[name] = result
    
    print(f"  {name}: {r2_str}, {time.time()-t1:.1f}s")

print("Training complete.")

### Reload sparse X for regression

In [ ]:
adata_disk = ad.read_h5ad(HSC_PATH, backed="r")

for name in ["CMP", "Mono"]:
    idx = cell_data[name]["idx"]
    X_sparse = sp.csr_matrix(adata_disk.X[idx])
    var_df = pd.DataFrame(index=var_names)
    
    adata_obj = ad.AnnData(X=X_sparse, obs=cell_data[name]["obs"].copy(), var=var_df)
    adata_obj.obsm["X_pca"] = cell_data[name]["pca"]
    adata_obj.obsm["cell_archetype_weights"] = cell_data[name]["weights"]
    adata_obj.obsm["archetype_distances"] = cell_data[name]["arch_dist"]
    for k, v in cell_data[name]["uns"].items():
        adata_obj.uns[k] = v
    pc.tl.assign_archetypes(adata_obj)
    cell_data[name]["adata"] = adata_obj
    print(f"  {name}: {adata_obj.shape}")

del adata_disk
gc.collect()

adata_cmp = cell_data["CMP"]["adata"]
adata_mono = cell_data["Mono"]["adata"]
print("Full AnnData objects rebuilt.")

## 3. Simplex Regression

Feature-level Scheffé polynomial regression on the simplex. This identifies genes whose expression is explained by archetype composition.

In [ ]:
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    t1 = time.time()
    reg = pc.tl.feature_simplex_regression(
        ad_obj, max_degree=2, n_bootstrap=0, robust_se=True,
    )
    elapsed = time.time() - t1
    median_r2 = np.median(reg["r_squared_degree1"])
    top_genes = np.argsort(reg["r_squared_degree1"])[-5:][::-1]
    top_names = [reg["feature_names"][i] for i in top_genes]
    top_r2s = reg["r_squared_degree1"][top_genes]
    
    print(f"\n{name} — median R2={median_r2:.3f}, {elapsed:.1f}s")
    print(f"  Top 5 genes:")
    for g, r in zip(top_names, top_r2s):
        print(f"    {g}: R2={r:.3f}")

## 4. Pattern Classification

Classify each gene's response pattern across the simplex (e.g., vertex-driven, edge-enriched, flat).

In [ ]:
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    patterns = pc.tl.classify_feature_patterns(ad_obj)
    counts = patterns["pattern_counts"]
    print(f"{name} pattern counts: {dict(counts)}")

In [ ]:
# Cell: GMM fitting
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    t1 = time.time()
    gmm_result = pc.tl.feature_simplex_decomposition(
        ad_obj, characterize_features=True,
    )
    elapsed = time.time() - t1
    print(f"{name}: optimal={gmm_result['n_components_optimal']}, "
        f"stable={gmm_result['n_components_stable']}, {elapsed:.1f}s")
    print(f"  Stability: {gmm_result['component_stability_scores']}")
    print(f"  Archetype map: {gmm_result['component_archetype_map']}")

In [ ]:
# Cell: GMM viz — component scatter
pc.pl.component_scatter(adata_cmp, show=True)

In [ ]:
# Cell: GMM viz — BIC curve
pc.pl.gmm_bic_curve(adata_cmp, show=True)

In [ ]:
# Cell: GMM viz — component feature heatmap
pc.pl.component_heatmap(adata_cmp, top_n=30, show=True)

In [ ]:
# Cell: GMM viz — stability bar plot
pc.pl.component_stability(adata_cmp, show=True)

## 5. Archetype Comparison (v0.5.0)

Three complementary comparison approaches:
- **MMD**: Kernel-based distribution distance between archetype populations
- **Feature similarity**: Silhouette + Spearman rank correlation on β vectors  
- **Wald contrasts**: Gene-level differential testing between archetype pairs

### 5a. Within-fit MMD

In [ ]:
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    mmd_result = pc.tl.archetype_mmd(ad_obj, n_permutations=100)
    mask = ~np.eye(K, dtype=bool)
    mmd_vals = mmd_result["mmd_matrix"][mask]
    print(f"{name} MMD range: {mmd_vals.min():.4f} – {mmd_vals.max():.4f}")
    print(f"  p-values (off-diag): {mmd_result['pvalue_matrix'][mask]}")

### 5b. Between-fit MMD (CMP vs Mono)

In [ ]:
mmd_between = pc.tl.archetype_mmd(adata_cmp, adata_mono, n_permutations=100)
print(f"Between-fit MMD matrix ({np.asarray(mmd_between['mmd_matrix']).shape}):")
print(np.array2string(np.asarray(mmd_between["mmd_matrix"]), precision=4))

### 5c. Feature similarity

In [ ]:
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    sim = pc.tl.archetype_feature_similarity(ad_obj)
    print(f"{name}: silhouette={sim['silhouette_overall']:.3f}")
    print(f"  Per-archetype: {np.array2string(np.asarray(sim['silhouette_per_archetype']), precision=3)}")

In [ ]:
# Between-fit feature similarity
sim_between = pc.tl.archetype_feature_similarity(adata_cmp, adata_mono)
print(f"Between-fit: n_shared={sim_between['n_shared_features']}")
print(f"Spearman matrix:")
print(np.array2string(np.asarray(sim_between["spearman_matrix"]), precision=3))

### 5d. Wald contrasts

In [ ]:
for name, ad_obj in [("CMP", adata_cmp), ("Mono", adata_mono)]:
    contrasts = pc.tl.archetype_contrasts(ad_obj)
    pair_summary = []
    for pair in contrasts["pairs"]:
        key = str(tuple(pair)) if not isinstance(pair, str) else pair
        n_sig = int(np.sum(np.asarray(contrasts["pvalues_fdr"][key]) < 0.05))
        pair_summary.append(f"{pair}: {n_sig} sig")
    print(f"{name} Wald contrasts:")
    for s in pair_summary:
        print(f"  {s}")

## 6. Flow Matching

Train a continuous normalizing flow to map CMP → CD14+ Monocyte in PCA space.

In [ ]:
# Build combined adata for flow
adata_cmp_flow = ad.AnnData(
    X=sp.csr_matrix((adata_cmp.n_obs, 1)),
    obs=adata_cmp.obs.copy(),
)
adata_cmp_flow.obsm["X_pca"] = cell_data["CMP"]["pca"]
adata_cmp_flow.obs["cell_type_label"] = "CMP"

adata_mono_flow = ad.AnnData(
    X=sp.csr_matrix((adata_mono.n_obs, 1)),
    obs=adata_mono.obs.copy(),
)
adata_mono_flow.obsm["X_pca"] = cell_data["Mono"]["pca"]
adata_mono_flow.obs["cell_type_label"] = "Mono"

adata_combined = ad.concat([adata_cmp_flow, adata_mono_flow])
print(f"Combined: {adata_combined.n_obs:,} cells")

In [ ]:
t1 = time.time()
flow_result = pc.tl.flow_within(
    adata_combined,
    source={"cell_type_label": "CMP"},
    target={"cell_type_label": "Mono"},
    pca_key="X_pca",
    hidden_dims=(64, 64, 64),
    n_epochs=FLOW_EPOCHS,
    batch_size=256,
    n_steps=50,
    device="cpu",
)
elapsed = time.time() - t1
print(f"Flow training: {elapsed:.1f}s")
print(f"MMD: {flow_result['mmd_before']:.4f} → {flow_result['mmd_after']:.4f}")

## 7. Visualization

All new v0.5.0 visualization functions. Plots are interactive (Plotly).

### Regression plots

In [ ]:
pc.pl.coefficient_heatmap(adata_cmp, top_n=30, show=True)

In [ ]:
pc.pl.r2_barplot(adata_cmp, top_n=30, show=True)

In [ ]:
pc.pl.regression_volcano(adata_cmp, show=True)

### Comparison plots

In [ ]:
pc.pl.mmd_heatmap(adata_cmp, show=True)

In [ ]:
pc.pl.feature_similarity_heatmap(adata_cmp, show=True)

In [ ]:
pc.pl.contrast_volcano(adata_cmp, pair=(0, 1), show=True)

### Flow plots

In [ ]:
pc.pl.flow_magnitude(adata_combined, flow_result, show=True)

In [ ]:
pc.pl.density_comparison(adata_combined, flow_result, show=True)

In [ ]:
pc.pl.velocity_quiver(adata_combined, flow_result, show=True)

## Summary

In [ ]:
elapsed_total = time.time() - t0
print(f"\n{'='*60}")
print(f"  v0.5.0 E2E Demo Complete")
print(f"{'='*60}")
print(f"  Total time: {elapsed_total:.1f}s")
print(f"  CMP: {adata_cmp.n_obs:,} cells, K={K}")
print(f"  Mono: {adata_mono.n_obs:,} cells, K={K}")
print(f"  Flow: MMD {flow_result['mmd_before']:.4f} → {flow_result['mmd_after']:.4f}")
print(f"{'='*60}")